# Lab 2: Zero-Data-Loss Failover

Atlas elects a new primary in about 5 seconds during a failover — a patch,
a scaling event, or a real hardware failure. With `w: "majority"` write
concern and retryable writes, the driver rides out that election
transparently. A continuous stream of simulated money-movement
transactions (wires, ACH, bill pay) keeps writing the whole time — the app
sees a brief latency blip, not an outage, and zero committed transactions
are lost.

Run this only against a sandbox/demo cluster — it triggers a real primary
failover on whatever's connected.

If you have the Grafana/Prometheus ops dashboard from `ops-dashboard/`
running, open it alongside this notebook — you'll see the primary and
secondary roles flip, and replication lag spike briefly, during the
election.

How to run it:
1. Run the setup and background-writer cells below.
2. While the live plot is running, go to Atlas UI → your cluster → ⋯ menu
   → Test Failover → Start. (Requires Project Cluster Manager+ access.)
3. Watch the plot: one short spike, then it flattens back out. Error count
   stays at zero.


In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pymongo[encryption]", "certifi", "python-dotenv", "requests", "matplotlib", "pandas"],
        check=True,
    )
    from getpass import getpass
    ATLAS_URI = os.environ.get("ATLAS_URI") or getpass("Atlas connection string (ATLAS_URI): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    ATLAS_URI = os.environ["ATLAS_URI"]

DEMO_DB = os.environ.get("DEMO_DB", "banking_demo")
import certifi
CA_FILE = certifi.where()
print("Environment:", "Colab" if IN_COLAB else "local", "| DB:", DEMO_DB)

import time, threading
from datetime import datetime, timezone
import matplotlib.pyplot as plt
from IPython.display import clear_output
from pymongo import MongoClient

# retryWrites=true + w="majority" is what lets the driver ride out the election.
# (Already the Atlas SRV default, but set explicitly here so it's visible.)
client = MongoClient(ATLAS_URI, retryWrites=True, w="majority", tlsCAFile=CA_FILE)
coll = client[DEMO_DB]["money_movement_stream"]
coll.drop()
print("Ready. Connected to", DEMO_DB)


In [ ]:
import random

# Background writer: one simulated money-movement transaction per second.
results = []  # list of dicts: {t, latency_ms, ok}
stop_event = threading.Event()

def make_transfer(seq):
    return {
        "seq": seq,
        "transfer_id": f"WT{seq:08d}",
        "type": random.choice(["wire", "ach", "bill_pay"]),
        "amount": round(random.uniform(50, 25000), 2),
        "ts": datetime.now(timezone.utc),
    }

def writer_loop():
    seq = 0
    while not stop_event.is_set():
        start = time.perf_counter()
        ok = True
        try:
            coll.insert_one(make_transfer(seq))
        except Exception:
            ok = False
        latency_ms = (time.perf_counter() - start) * 1000
        results.append({"t": time.time(), "latency_ms": latency_ms, "ok": ok})
        seq += 1
        time.sleep(1)

thread = threading.Thread(target=writer_loop, daemon=True)
thread.start()
print("Background writer started — 1 money-movement transaction/sec.")


### Now trigger the failover

Go to the Atlas UI and click **Test Failover** on this cluster. Then run the
next cell — it live-plots for ~90 seconds, which is plenty of time to click
the button and watch the result.

Prefer the CLI or Admin API instead of clicking through the UI? See the
commented cell after the plot.


In [ ]:
PLOT_SECONDS = 90
t0 = time.time()

while time.time() - t0 < PLOT_SECONDS:
    clear_output(wait=True)
    if results:
        xs = [r["t"] - t0 for r in results]
        ys = [r["latency_ms"] for r in results]
        errs_x = [r["t"] - t0 for r in results if not r["ok"]]
        errs_y = [r["latency_ms"] for r in results if not r["ok"]]

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(xs, ys, color="#2e7d32", linewidth=1.5, label="write latency (ms)")
        if errs_x:
            ax.scatter(errs_x, errs_y, color="#c62828", zorder=5, label="failed write")
        ax.set_xlabel("seconds since start")
        ax.set_ylabel("latency (ms)")
        ax.set_title(f"Live write latency — {len(results)} writes, "
                      f"{sum(1 for r in results if not r['ok'])} errors")
        ax.legend(loc="upper right")
        plt.show()
    time.sleep(1)

print("Done plotting. Writer thread is still running — stop it in the next cell.")


In [ ]:
# Optional automation instead of the manual UI click. Requires the Atlas CLI
# (https://www.mongodb.com/docs/atlas/cli/) installed and authenticated, or
# ATLAS_PUBLIC_KEY/ATLAS_PRIVATE_KEY set for `atlas` to pick up.
#
# import subprocess
# cluster_name = os.environ["ATLAS_CLUSTER_NAME"]
# project_id = os.environ["ATLAS_PROJECT_ID"]
# subprocess.run(
#     ["atlas", "clusters", "failover", cluster_name, "--projectId", project_id, "--force"],
#     check=True,
# )


In [ ]:
stop_event.set()
thread.join(timeout=5)

total = len(results)
errors = sum(1 for r in results if not r["ok"])
max_latency = max((r["latency_ms"] for r in results), default=0)
print(f"Total writes: {total} | Errors: {errors} | Max latency seen: {max_latency:.0f} ms")


### Why this matters

- Zero application code changed. This is default driver behavior with
  retryable writes — every current MongoDB driver enables this by default.
- The blip is the ~5-second election window, not downtime.
- Percona Server for MongoDB runs the same replica-set election mechanism —
  this isn't a capability gap. What Atlas adds is **Test Failover**: trigger
  and validate this exact scenario on demand, instead of finding out how
  your application handles it during a real incident. Atlas also runs the
  same election automatically for routine patches and scaling events, with
  no maintenance window or manual coordination required.
- If `ops-dashboard/` is running, replication lag and primary/secondary
  role changes show up there in real time during the election. Atlas gives
  you this visibility out of the box (Performance Advisor, Real-Time
  Performance Panel) — a self-managed deployment needs a DIY stack like the
  one in this repo to get the same view.
- If the cluster is multi-region, the same story extends to **Simulate
  Regional Outage** — a 1/3/7-day simulated outage of an entire region.
